In [7]:
import cv2
import json
import os
import torch
from ultralytics.models import YOLO

# --- CONFIG ---
VIDEO_FOLDER = "D:/GitHub/BaseballPitch/modules/pitcher_segmentation/finetuning_dataset/videos"
OUTPUT_BASE = "D:/GitHub/BaseballPitch/modules/pitcher_segmentation/finetuning_dataset"
SPLIT = "train"  # 'train' or 'val' or 'test'
PITCH_TYPE = "CH - Changeup"
POSE_MODEL = 'D:/GitHub/BaseballPitch/yolo26n-pose.pt'  # Detect people with pose keypoints
CONF = 0.25  # Confidence threshold for pose detection
IMG_SIZE = 1280
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# Frame skip: only label every Nth frame to speed up (1 = every frame)
FRAME_SKIP = 1  # Label every Nth frame
MAX_VIDEOS = None  # Set to a number to limit videos, or None for all
# Progress tracking file — records which videos are fully labeled
PROGRESS_FILE = f"{OUTPUT_BASE}/labeling_progress.json"
# --------------

In [ ]:
class SemiAutoLabeler:
    def __init__(self):
        self.pose_model = YOLO(POSE_MODEL)
        self.pose_model.to(DEVICE)
        self.current_frame = None
        self.current_frame_num = 0
        self.video_name = ""
        self.labeled_count = 0
        self.skipped_count = 0
        self.deleted_count = 0
        self.detected_boxes = []
        self.detected_keypoints = []
        self.mouse_x = 0
        self.mouse_y = 0
        self.progress = self._load_progress()

    def _load_progress(self):
        """Load labeling progress from JSON file"""
        if os.path.exists(PROGRESS_FILE):
            with open(PROGRESS_FILE, "r") as f:
                return json.load(f)
        return {}

    def _save_progress(self):
        """Save labeling progress to JSON file"""
        os.makedirs(os.path.dirname(PROGRESS_FILE), exist_ok=True)
        with open(PROGRESS_FILE, "w") as f:
            json.dump(self.progress, f, indent=2)

    def _progress_key(self, video_name):
        """Generate a unique progress key for the current split/pitch/video"""
        return f"{SPLIT}/{PITCH_TYPE}/{video_name}"

    def mouse_callback(self, event, x, y, flags, param):
        """Handle mouse events to track position and clicks"""
        # Always track mouse position
        self.mouse_x = x
        self.mouse_y = y

        if event == cv2.EVENT_LBUTTONDOWN:
            for idx, box in enumerate(self.detected_boxes):
                x1, y1, x2, y2 = box
                if x1 <= x <= x2 and y1 <= y <= y2:
                    self.save_label(idx)
                    break

    def save_label(self, box):
        """Save YOLO format label for selected box"""
        if self.current_frame is None:
            return
                self.save_label(selected_box)
    def save_label(self, detection_idx):
        """Save YOLO-pose format label (box + keypoints) for selected detection"""

        # Convert to YOLO format (normalized center x, y, width, height)
        x_center = ((x1 + x2) / 2) / w
        y_center = ((y1 + y2) / 2) / h
        x1, y1, x2, y2 = self.detected_boxes[detection_idx]
        kpts = self.detected_keypoints[detection_idx]

        # Convert to YOLO format (normalized center x, y, width, height)
        x_center = ((x1 + x2) / 2) / w
        y_center = ((y1 + y2) / 2) / h
        box_width = (x2 - x1) / w
        box_height = (y2 - y1) / h

        # Build keypoint string: x y v for each of 17 COCO keypoints
        kpt_parts = []
        for kx, ky, kconf in kpts:
            if kconf > 0.5:
                kpt_parts.extend([f"{kx / w:.6f}", f"{ky / h:.6f}", "2"])
            else:
                kpt_parts.extend(["0.000000", "0.000000", "0"])

        # Save image
        img_name = f"{self.video_name}_{self.current_frame_num:04d}.jpg"
        img_path = f"{OUTPUT_BASE}/images/{SPLIT}/{PITCH_TYPE}/{img_name}"
        cv2.imwrite(img_path, self.current_frame)

        # Save label in YOLO-pose format: class box keypoints
        label_path = (
            f"{OUTPUT_BASE}/labels/{SPLIT}/{PITCH_TYPE}/"
            f"{self.video_name}_{self.current_frame_num:04d}.txt"
        )
        with open(label_path, "w") as f:
            f.write(
                f"0 {x_center:.6f} {y_center:.6f} "
                f"{box_width:.6f} {box_height:.6f} "
                f"{' '.join(kpt_parts)}\n"
            )

        self.labeled_count += 1
        print(
            f"✓ Labeled frame {self.current_frame_num} "
            f"({self.labeled_count} total)"
        )
        )
    def save_frame_no_label(self):
        """Save frame without label (negative example)"""
        if self.current_frame is None:
            return

        # Save image only
        img_name = f"{self.video_name}_{self.current_frame_num:04d}.jpg"
        img_path = f"{OUTPUT_BASE}/images/{SPLIT}/{PITCH_TYPE}/{img_name}"
        cv2.imwrite(img_path, self.current_frame)

        # Create empty label file
        label_path = (
            f"{OUTPUT_BASE}/labels/{SPLIT}/{PITCH_TYPE}/"
            f"{self.video_name}_{self.current_frame_num:04d}.txt"
        )
        open(label_path, "w").close()  # Empty file = no objects

        self.skipped_count += 1
        print(
            f"⏭️  Saved frame {self.current_frame_num} without label "
            f"({self.skipped_count} total)"
        )

    def label_video(self, video_path):
        """Label all frames in a video"""
        self.video_name = os.path.splitext(os.path.basename(video_path))[0]
        key = self._progress_key(self.video_name)

        # Skip if already fully labeled according to progress log
        if key in self.progress and self.progress[key].get("status") == "done":
            print(
                f"⏭️  Skipping {self.video_name} "
                f"(already completed in progress log)"
            )
            return True

        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        print(f"\n{'='*60}")
        print(f"Video: {self.video_name}")
        print(f"Split: {SPLIT}")
        print(f"Total frames: {total_frames} (labeling every {FRAME_SKIP})")
        print(f"Using device: {DEVICE}")
        print(f"{'='*60}")
        print("\nControls:")
        print("  - Click on pitcher OR press 'A' to select first bounding box")
        print("  - SPACE/RIGHT ARROW - Save frame without pitcher label")
        print("  - D - Delete frame (don't save)")
        print("  - Q - Quit labeling session")
        print("  - N - Skip to next video")

        cv2.namedWindow("Semi-Auto Labeling")
        cv2.setMouseCallback("Semi-Auto Labeling", self.mouse_callback)

        frames_to_label = list(range(0, total_frames, FRAME_SKIP))

        # For partially labeled videos, figure out which frames are already done
        label_dir = f"{OUTPUT_BASE}/labels/{SPLIT}/{PITCH_TYPE}"
        already_labeled = set()
        if os.path.exists(label_dir):
            for f in os.listdir(label_dir):
                if f.startswith(self.video_name) and f.endswith('.txt'):
                    # Extract frame number from e.g. videoname_0042.txt
                    stem = os.path.splitext(f)[0]
                    parts = stem.rsplit('_', 1)
                    if len(parts) == 2 and parts[1].isdigit():
                        already_labeled.add(int(parts[1]))

        remaining = [f for f in frames_to_label if f not in already_labeled]
        if already_labeled:
            print(
                f"Resuming: {len(already_labeled)} frames already labeled, "
                f"{len(remaining)} remaining"
            )

        for frame_idx in remaining:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if not ret:
                break
            # Extract bounding boxes and keypoints in pixel coordinates
            self.current_frame = frame.copy()
            self.detected_keypoints = []
            result = pose_results[0]
            boxes_obj = result.boxes
            kpts_obj = result.keypoints
            if boxes_obj is not None and len(boxes_obj) > 0:
                # Get boxes in x1,y1,x2,y2 format
                boxes = boxes_obj.xyxy
                if isinstance(boxes, torch.Tensor):
                    boxes = boxes.cpu().numpy()

                # Get keypoints: [N, 17, 3] -> (x, y, conf) per keypoint
                kpts_data = kpts_obj.data
                if isinstance(kpts_data, torch.Tensor):
                    kpts_data = kpts_data.cpu().numpy()
                if isinstance(boxes, torch.Tensor):
                for i, box in enumerate(boxes):
                    x1, y1, x2, y2 = map(int, box[:4])
                    self.detected_boxes.append((x1, y1, x2, y2))
                    self.detected_keypoints.append(kpts_data[i])

            while True:
                display_frame = self.current_frame.copy()

                # Draw all detected bounding boxes
                for i, (x1, y1, x2, y2) in enumerate(self.detected_boxes):
                    cv2.rectangle(
                        display_frame, (x1, y1), (x2, y2), (0, 255, 0), 2
                    )
                    cv2.putText(
                        display_frame, f"Person {i+1}", (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2
                    )

                # Add info overlay
                progress_idx = remaining.index(frame_idx) + 1
                info_text = (
                    f"Frame {self.current_frame_num}/{total_frames} | "
                    f"Progress: {progress_idx}/{len(remaining)}"
                    f"Frame {self.current_frame_num}/{total_frames} | "
                cv2.putText(
                    display_frame, info_text, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2
                )
                cv2.putText(
                    display_frame, info_text, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 1
                )

                if self.detected_boxes:
                    instruction = (
                        "Click pitcher or press 'A' | "
                        "SPACE=no label | D=delete"
                    )
                else:
                    instruction = (
                        "No people detected - "
                        "SPACE=no label | D=delete"
                    )
                cv2.putText(
                    display_frame, instruction, (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2
                )
                cv2.putText(
                    display_frame, instruction, (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1
                )

                        selected_idx = None
                        for idx, box in enumerate(self.detected_boxes):
                key = cv2.waitKeyEx(1)

                # Press 'A' to select bounding box under mouse cursor
                if key in (ord('a'), ord('A')):
                                selected_idx = idx
                        # Check if mouse is over any bounding box
                        selected_box = None
                        if selected_idx is not None:
                            self.save_label(selected_idx)
                            in_x = x1 <= self.mouse_x <= x2
                            in_y = y1 <= self.mouse_y <= y2
                            if in_x and in_y:
                                selected_box = box
                                break

                        if selected_box:
                            self.save_label(selected_box)
                            break
                        else:
                            print(
                                "⚠️  Move mouse over a bounding box "
                                "to select with 'A'"
                            )
                    else:
                        print("⚠️  No bounding boxes detected")

                # Save frame without label (negative example)
                # Space or Right arrow
                elif key in (32, 2555904, ord('s'), ord('S')):
                    self.save_frame_no_label()
                    break

                # Delete frame (don't save)
                elif key in (ord('d'), ord('D')):
                    self.deleted_count += 1
                    print(
                        f"🗑️  Deleted frame {self.current_frame_num} "
                        f"({self.deleted_count} total)"
                    )
                    break

                # Quit
                elif key in (ord('q'), ord('Q'), 27):
                    cap.release()
                    cv2.destroyAllWindows()
                    return False

                # Next video
                elif key in (ord('n'), ord('N')):
                    cap.release()
                    return True
                    cap.release()
                # Check if label was just saved (frame exists)
                img_path = (
                    f"{OUTPUT_BASE}/images/{SPLIT}/{PITCH_TYPE}/"
                    f"{self.video_name}_{self.current_frame_num:04d}.jpg"
                )
                if os.path.exists(img_path):
                    break

        cap.release()

        # Mark video as done in progress log
        self.progress[key] = {
            "status": "done",
            "labeled": self.labeled_count,
            "skipped": self.skipped_count,
            "deleted": self.deleted_count,
        }
        self._save_progress()
        return True

    def run(self):
        """Main labeling loop"""
        videos = sorted([
            f for f in os.listdir(f"{VIDEO_FOLDER}/{SPLIT}/{PITCH_TYPE}")
            if f.endswith('.mp4')
        ])
            if f.endswith('.mp4')
        # Limit videos if MAX_VIDEOS is set
        if MAX_VIDEOS is not None:
            videos = videos[:MAX_VIDEOS]

        if not videos:
            print(f"No videos found in {VIDEO_FOLDER}/{SPLIT}/{PITCH_TYPE}")
            return
            print(f"No videos found in {VIDEO_FOLDER}/{SPLIT}/{PITCH_TYPE}")
        # Count already-completed videos
        done_count = sum(
            1 for v in videos
            if self.progress.get(self._progress_key(os.path.splitext(v)[0]), {}).get("status") == "done"
        )
        print(f"Found {len(videos)} videos ({done_count} already completed)")
        print("\nStarting semi-automatic labeling session...")

        for i, video in enumerate(videos, 1):
            print(f"\n[Video {i}/{len(videos)}]")
            video_path = os.path.join(f"{VIDEO_FOLDER}/{SPLIT}/{PITCH_TYPE}", video)

            if not self.label_video(video_path):
                break  # User quit
        cv2.destroyAllWindows()
        print("\n" + "=" * 60)
        print("Labeling session complete!")
        print(f"Labeled (with pitcher): {self.labeled_count} frames")
        print(f"Saved (without pitcher): {self.skipped_count} frames")
        print(f"Deleted: {self.deleted_count} frames")
        print("=" * 60)

            "1. Review labels: python "
        print("\nNext steps:")            "modules/release_detection/scripts/pitcher_training.py"

            "modules/release_detection/scripts/pitcher_review_labels.py"
        print(            "2. Train model: python "

        )
            "1. Review labels: python "        )

        print(
            "modules/release_detection/scripts/pitcher_review_labels.py"        print(

            "2. Train model: python "
        )            "modules/release_detection/scripts/pitcher_training.py"

In [6]:
labeler = SemiAutoLabeler()
labeler.run()

Found 21 videos (20 already completed)

Starting semi-automatic labeling session...

[Video 1/21]
⏭️  Skipping PitchType-CH_Zone-11_PlayID-cd61c6db-63f7-341d-9928-517e2778d741_Date-2025-09-02 (already completed in progress log)

[Video 2/21]
⏭️  Skipping PitchType-CH_Zone-11_PlayID-e5658640-5139-3bcc-b490-4ba9332bde5c_Date-2025-09-02 (already completed in progress log)

[Video 3/21]
⏭️  Skipping PitchType-CH_Zone-11_PlayID-f1529eed-27cf-3798-beef-64ea0eaca713_Date-2025-09-13 (already completed in progress log)

[Video 4/21]
⏭️  Skipping PitchType-CH_Zone-13_PlayID-3b4a2254-1988-3053-929c-cf058e047375_Date-2025-08-22 (already completed in progress log)

[Video 5/21]
⏭️  Skipping PitchType-CH_Zone-13_PlayID-9a15fc57-fcd9-3050-967f-1f4ffa875792_Date-2025-08-22 (already completed in progress log)

[Video 6/21]
⏭️  Skipping PitchType-CH_Zone-13_PlayID-9ed1178f-cb29-3749-9c22-dc58cea7d176_Date-2025-08-22 (already completed in progress log)

[Video 7/21]
⏭️  Skipping PitchType-CH_Zone-13_Play